# DeepVerse-Python Digital Twin Beam Prediction Demo

## Introduction

In this notebook, we present a digital twin beam prediciton demo of the [DeepVerse 6G](https://deepverse6g.net/) dataset.
- We train a position-based beam prediction NN on the digital twin (synthetic) data.
- The trained model is tested on the corresponding real-world data.

See more detailed [API documentation](https://deepverse6g.net/documentation)\
See more detailed [DeepVerse generator tutorial](https://deepverse6g.net/tutorials)

Please cite the following paper if you use this notebook (parts or modified version) in your research project.

In [ ]:
'''
@inproceedings{jiang2023digital,
  title={Digital twin based beam prediction: Can we train in the digital world and deploy in reality?},
  author={Jiang, Shuaifeng and Alkhateeb, Ahmed},
  booktitle={2023 IEEE International Conference on Communications Workshops (ICC Workshops)},
  pages={36--41},
  year={2023},
  organization={IEEE}
}
'''

## Installation

In [ ]:
# Installation needed starting from a fresh conda env
# This may not produce a torch installation supporting cuda and gpu
# %pip install torch numpy pandas tqdm scikit-learn requests deepverse scipy

## Import

In [21]:
import os
import sys
from scipy.io import loadmat
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import zipfile
import requests
import shutil
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from deepverse import ParameterManager, Dataset

## Digital twin synthetic data preparation

### Download DeepVerse data

Here we download the DeepVerse (synthetic) data. We use the DT1 scenario, which incoporates the LiDAR, RGB images, wireless, and position data modalities. Since this demo focuses on the position-based beam prediction application, we only download and use the wireless data modality.

The scenario folder follows the below structure
```
DeepVerse-main/
├─ scenarios/
│  ├─ DT1/
│  │  ├─ wireless/
│  │  │  ├─ ...
│  │  │  ├─ params.mat
│  │  ├─ param/
│  │  │  ├─ config.m
│  │  ├─ scenario1.csv
```

In [36]:
def download_and_unzip(url, zip_path, extract_to):
    response = requests.get(url, stream=True)
    response.raise_for_status()
    
    total = int(response.headers.get('content-length', 0))
    with open(zip_path, 'wb') as f, tqdm(
        desc=f"Downloading: {zip_path.name}",
        total=total,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for chunk in response.iter_content(chunk_size=8192):
            size = f.write(chunk)
            bar.update(size)

    print(f"Infalting: {zip_path}")
    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_to)
    except zipfile.BadZipFile:
        print(f"Error: {zip_path} is not a valid zip file!")
        return

    print(f"Removing zip: {zip_path}")
    zip_path.unlink()

In [37]:
# Set up directories
scenario_name = 'DT1'
scenario_dir = Path(f"scenarios/{scenario_name}")
scenario_dir.mkdir(parents=True, exist_ok=True)


# Download and extract wireless data
print("Preparing wireless data")
download_and_unzip(
    "https://www.dropbox.com/scl/fi/esl0jd1idkwarmk9sr1xz/wireless.zip?rlkey=qhwdiu07oasfn1h9pa5xfjv0i&st=pjgypntm&dl=1",
    scenario_dir / "wireless.zip",
    scenario_dir
)

# Download and extract parameter files
print("Preparing param files")
param_dir = scenario_dir / "param"
param_dir.mkdir(exist_ok=True)
download_and_unzip(
    "https://www.dropbox.com/scl/fo/9sfd6u8912l7o407fqi30/AN2NIxPUrXvMEVjImsHmX2g?rlkey=qqxzkohhnmgjgz2abf6cvb32h&st=0kbpp7v4&dl=1",
    scenario_dir / "param.zip",
    param_dir
)

# Copy wireless params.mat file to wireless folder
wireless_dir = scenario_dir / "wireless"
shutil.copy(param_dir / "params.mat", wireless_dir / "params.mat")
shutil.copy(param_dir / "scenario1.csv", scenario_dir / "scenario1.csv")

print('Completed')

Preparing wireless data


Downloading: wireless.zip: 100%|██████████| 129M/129M [00:01<00:00, 84.4MB/s] 


Infalting: scenarios\DT1\wireless.zip
Removing zip: scenarios\DT1\wireless.zip
Preparing param files


Downloading: param.zip: 100%|██████████| 3.19M/3.19M [00:00<00:00, 13.5MB/s]

Infalting: scenarios\DT1\param.zip
Removing zip: scenarios\DT1\param.zip
Completed


### DeepVerse dataset generation

In [38]:
# path to the configuration file
config_path = f'scenarios/{scenario_name}/param/config.m'

# initialize ParameterManager and load parameters
param_manager = ParameterManager(config_path)
param_manager.params["scenes"] = list(range(2411))
param_manager.params["radar"]["enable"] = False
param_manager.params["comm"]["enable"] = True
param_manager.params["camera"] = False
param_manager.params["lidar"] = False
param_manager.params["position"] = True

# generate a dataset
dataset = Dataset(param_manager)

Generating mobility dataset: ⏳ In progress
Generating mobility dataset: ✅ Completed (0.00s)
Generating comm dataset: ⏳ In progress


Generating comm dataset: ✅ Completed (70.26s)


### Data postprcessing

The generated DeepVerse dataset consists of:
- The communication channel between the basestation and the user
- The position of the communication user

In the following blocks we apply postprocessing on this data:
- We apply beamforming (from a beam codebook) to the communication channel to get the communication beam power
- We will then save the postprocessed data: beam power and user position.

#### Simple beam-steering codebook

In [39]:
def beam_steering_codebook(angles, num_z, num_x):
    d = 0.5
    k_z = np.arange(num_z)
    k_x = np.arange(num_x)
    
    codebook = []
    
    for beam_idx in range(angles.shape[0]):
        z_angle = angles[beam_idx, 0]
        x_angle = angles[beam_idx, 1]
        bf_vector_z = np.exp(1j * 2 * np.pi * k_z * d * np.cos(np.radians(z_angle)))
        bf_vector_x = np.exp(1j * 2 * np.pi * k_x * d * np.cos(np.radians(x_angle)))
        bf_vector = np.outer(bf_vector_z, bf_vector_x).flatten()
        codebook.append(bf_vector)
        
    return np.stack(codebook, axis=0)

# construct beam steering codebook
x_angles = loadmat(f'scenarios/{scenario_name}/param/beam_angles.mat')['beam_angles'].squeeze() * 180 / np.pi
num_angles = x_angles.size

# x_angles = np.flip(x_angles)
z_angles = np.full(num_angles, 90)
beam_angles = np.column_stack((z_angles, x_angles))
codebook =  beam_steering_codebook(beam_angles, 1, 16)

#### Save communication beam power and position

In [40]:
data_dir = 'data'
os.makedirs(data_dir, exist_ok=True)

# apply codebook to bs-ue comm channel
beam_power = []
position = []
num_scene = len(dataset.params['scenes'])
for i in range(num_scene):
    channel = dataset.get_sample('comm-ue', index=i, bs_idx=0, ue_idx=0).coeffs
    beam_power_ = (np.abs(codebook @ np.squeeze(channel, 0))**2).sum(-1)
    beam_power.append(beam_power_)
    position.append(dataset.get_sample('loc-ue', index=i, bs_idx=0, ue_idx=0))
np.save(f'{data_dir}/dt_beam_power.npy', np.stack(beam_power, 0))
np.save(f'{data_dir}/pos.npy', np.stack(position, 0))

## Real world data preparation

In the following block
- Download the [Scenario 1](https://www.deepsense6g.net/scenario-1/) data from the [DeepSense 6G dataset](https://www.deepsense6g.net). This scenario is the real-world counterpart of the DeepVerse DT1.
- Extract and save the positon of the user veichle and the communication beam power at the base station

### Download real world data

In [41]:
# Download and extract the real world data
os.makedirs('scenarios', exist_ok=True)
print("Preparing real world data")
download_and_unzip(
    "https://www.dropbox.com/scl/fi/qmzvp20sa8wjb9a6y1qxp/deepsense_scenario1.zip?rlkey=qume5rnyj303izovj6op3npl2&st=dofqyj3r&dl=1",
    Path("scenarios/scenario1.zip"),
    Path("scenarios")
)

Preparing real world data


Downloading: scenario1.zip: 100%|██████████| 425M/425M [00:05<00:00, 75.2MB/s] 


Infalting: scenarios\scenario1.zip
Removing zip: scenarios\scenario1.zip


### Postprocess real world data

In [42]:
rw_data_dir = 'scenarios/scenario1'

data_csv = pd.read_csv(f'{rw_data_dir}/scenario1.csv')
rw_beam_power = []
for a in data_csv['unit1_comm1']:
    rw_beam_power.append(np.loadtxt(os.path.join(rw_data_dir, a)))
rw_beam_power = np.stack(rw_beam_power, 0)
np.save(f'{data_dir}/rw_beam_power.npy', rw_beam_power)

## Create Deep learning training and testing datasets

Here, we divide the full dataset to training and testing datasets.

In [43]:
# Split without data leakage based on seq_index
unique_seq_indices = data_csv['seq_index'].unique()
train_seqs, test_seqs = train_test_split(unique_seq_indices, test_size=0.2, random_state=42)

data_csv = data_csv[['scene', 'seq_index']]

train_df = data_csv[data_csv['seq_index'].isin(train_seqs)]
test_df = data_csv[data_csv['seq_index'].isin(test_seqs)]

# Save the split CSVs
train_df.to_csv( f'{data_dir}/{scenario_name}_train.csv', index=False)
test_df.to_csv(f'{data_dir}/{scenario_name}_test.csv', index=False)

## Deep learning demo

In the following blocks we train a neural network for position-aided beam prediction using digital twin
- The input of the neural network is the position of the user vehicle
- The ground-truth output of the neural network is a one-hot vector for the optimal beam index
- The loss function for training is the cross-entropy
- We train the NN on the digital twin synthetic data
- We test the performance (beam prediction with top-k accuracy) of the NN on the real-world data.

### Create dataloader

In [45]:
class DataFeed(Dataset):
    def __init__(self, pos_path, rw_beam_power_path, dt_beam_power_path, data_csv_path):
        pos = np.load(pos_path, allow_pickle=True)[:, :-1]
        rw_beam_power = np.load(rw_beam_power_path, allow_pickle=True)
        dt_beam_power = np.load(dt_beam_power_path, allow_pickle=True)

        rw_beam_power = rw_beam_power[:, 1::4]
        dt_beam_power = dt_beam_power[:, 1::4]
        rw_best_beam = np.argmax(rw_beam_power, 1)
        dt_best_beam = np.argmax(dt_beam_power, 1)

        polar_anlge = np.arctan2(pos[:, 1], pos[:, 0]) / np.pi
        polar_distance = np.sqrt(pos[:, 1]**2 + pos[:, 0]**2) / np.sqrt(24**2+28**2)
        pos = pos / 30.
        pos_ext = np.concatenate([pos, np.stack([polar_anlge, polar_distance], -1)], -1)

        data_csv = pd.read_csv(data_csv_path)
        self.scene_idx = torch.from_numpy(np.stack([i for i in data_csv['scene']])).long()
        self.pos_ext = torch.from_numpy(pos_ext[self.scene_idx]).float()
        self.rw_beam_power = torch.from_numpy(rw_beam_power[self.scene_idx]).float()
        self.dt_beam_power = torch.from_numpy(dt_beam_power[self.scene_idx]).float()
        self.rw_best_beam = torch.from_numpy(rw_best_beam[self.scene_idx]).long()
        self.dt_best_beam = torch.from_numpy(dt_best_beam[self.scene_idx]).long()
    
    def __len__(self):
        return int(self.rw_best_beam.shape[0])
    
    def __getitem__(self, idx):
        pos_ext = self.pos_ext[idx, ...]
        rw_beam_power = self.rw_beam_power[idx, ...]
        dt_beam_power = self.dt_beam_power[idx, ...]
        rw_best_beam = self.rw_best_beam[idx, ...]
        dt_best_beam = self.dt_best_beam[idx, ...]
        
        return pos_ext, rw_best_beam, dt_best_beam, rw_beam_power, dt_beam_power

In [46]:
real_beam_pwr_path = 'data/real_beam_pwr.mat'
real_pos_path = 'data/ue_relative_pos.mat'
synth_beam_pwr_path = 'data/synth_beam_power_uniform.mat'
synth_pos_path = 'data/pos.npyt'

pos_path = 'data/pos.npy'
rw_beam_power_path = 'data/rw_beam_power.npy'
dt_beam_power_path = 'data/dt_beam_power.npy'
train_csv_path = 'data/DT1_train.csv'
test_csv_path = 'data/DT1_test.csv'

torch.manual_seed(1115)

num_epoch = 80
batch_size = 32
val_batch_size =128

all_acc = []
all_pwr = []
train_loader = DataLoader(DataFeed(pos_path, rw_beam_power_path, dt_beam_power_path, train_csv_path), batch_size=batch_size, shuffle=True)
test_loader = DataLoader(DataFeed(pos_path, rw_beam_power_path, dt_beam_power_path, test_csv_path), batch_size=batch_size, shuffle=True)

### Deep learning model

In [47]:
class FullyConnected(nn.Module):
    def __init__(self, num_classes, hidden_size=256):
        super(FullyConnected, self).__init__()
        self.fc1 = nn.Linear(4, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, hidden_size)
        self.classifier = nn.Linear(hidden_size, num_classes)

        self.relu = nn.ReLU(inplace=True)
        self.dropout1 = nn.Dropout(0.5)

    def forward(self, x):
        y = self.fc1(x)
        y = self.relu(y)
        y = self.fc2(y)
        y = self.relu(y)
        y = self.fc3(y)
        y = self.relu(y)
        y = self.classifier(y)
        return y

### Train model

In [48]:
num_classes=16
num_epoch=80
lr=1e-2

# check gpu acceleration availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Assuming that we are on a CUDA machine, this should print a CUDA device:
print(device)
# Instantiate the model
net = FullyConnected(num_classes)
# send model to GPU
net.to(device)

# set up loss function and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(net.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[20, 40, 60], gamma=0.2
)

# train model
for epoch in range(num_epoch):  # loop over the dataset multiple times
    net.train()
    running_loss = 0.0
    running_acc = 1.0
    with tqdm(train_loader, unit="batch", file=sys.stdout) as tepoch:
        for i, (pos, _, dt_best_beam, _, dt_beam_power) in enumerate(tepoch, 0):
            tepoch.set_description(f"Epoch {epoch}")

            pos = pos.to(device)
            dt_best_beam = dt_best_beam.to(device)
            optimizer.zero_grad()

            # forward + backward + optimize
            outputs = net(pos)
            loss = criterion(outputs, dt_best_beam)
            prediction = torch.argmax(outputs, dim=-1)
            acc = (prediction == dt_best_beam).float().mean().item()
            loss.backward()
            optimizer.step()

            # print statistics
            running_loss = (loss.item() + i * running_loss) / (i + 1)
            running_acc = (acc + i * running_acc) / (i + 1)
            tepoch.set_postfix({"loss": running_loss, "accuracy":running_acc})
        scheduler.step()

    os.makedirs('checkpoints', exist_ok=True)
    torch.save(net.state_dict(), 'checkpoints/test1')
print("Finished Training")

cuda
Epoch 79: 100%|██████████| 60/60 [00:00<00:00, 144.34batch/s, loss=0.252, accuracy=0.904]
Finished Training


### Test model

In [49]:
net = FullyConnected(num_classes)
net.load_state_dict(torch.load('checkpoints/test1'))
net.to(device)

# test
predictions = []
raw_predictions = []
true_label = []
net.eval()
with torch.no_grad():
    total = 0
    top1_correct = 0
    top2_correct = 0
    top3_correct = 0
    top5_correct = 0
    top1_pwr = 0
    top2_pwr = 0
    top3_pwr = 0
    top5_pwr = 0
    test_loss = 0
    for (pos, rw_best_beam, _, rw_beam_power, _) in test_loader:
        pos = pos.to(device)
        rw_best_beam = rw_best_beam.to(device)
        optimizer.zero_grad()
        
        outputs = net(pos)

        test_loss += nn.CrossEntropyLoss(reduction="sum")(
            outputs.view(-1, num_classes), rw_best_beam.flatten()
        ).item()
        total += rw_best_beam.cpu().numpy().size
        prediction = torch.argmax(outputs, dim=-1)
        top1_correct += torch.sum(prediction == rw_best_beam, dim=-1).cpu().numpy()

        _, idx = torch.topk(outputs, 5, dim=-1)
        idx = idx.cpu().numpy()
        rw_best_beam = rw_best_beam.cpu().numpy()
        for j in range(rw_best_beam.shape[0]):
            top2_correct += np.isin(rw_best_beam[j], idx[j, :2]).sum()
            top3_correct += np.isin(rw_best_beam[j], idx[j, :3]).sum()
            top5_correct += np.isin(rw_best_beam[j], idx[j, :5]).sum()


        predictions.append(prediction.cpu().numpy())
        raw_predictions.append(outputs.cpu().numpy())
        true_label.append(rw_best_beam)

    test_loss /= float(total)
    test_top1_acc = top1_correct / float(total)
    test_top2_acc = top2_correct / float(total)
    test_top3_acc = top3_correct / float(total)
    test_top5_acc = top5_correct / float(total)

    test_acc = np.asarray([
        test_top1_acc,
        test_top2_acc,
        test_top3_acc,
        test_top5_acc
    ])

    print(test_acc)

[0.7049505  0.96039604 0.98217822 0.9960396 ]
